# Forecast-Feature Linear Regression Baselines

This notebook trains two same-time linear regression baselines using forecast boundary values:

`boundary_forecast_values(t) -> price(t)`

The two feature sets compare `Wind_And_Solar` against `Wind_Power + Photovoltaic`.

In [ ]:
from pathlib import Path
import os


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "output/clean_datasets/price_prediction_forecast_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find output/clean_datasets/price_prediction_forecast_dataset.csv")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_forecast_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/baseline_results/price_prediction_forecast"

AGGREGATE_FEATURE_COLUMNS = [
    "System_Load",
    "Wind_And_Solar",
    "Tie_Line",
    "Hydropower",
    "Non-Marketized_Unit",
]
COMPONENT_FEATURE_COLUMNS = [
    "System_Load",
    "Wind_Power",
    "Photovoltaic",
    "Tie_Line",
    "Hydropower",
    "Non-Marketized_Unit",
]
ALL_FEATURE_COLUMNS = list(dict.fromkeys(AGGREGATE_FEATURE_COLUMNS + COMPONENT_FEATURE_COLUMNS))
TARGET_COLUMN = "price"

AGGREGATE_MODEL_NAME = "Forecast Linear Regression (Wind_And_Solar)"
COMPONENT_MODEL_NAME = "Forecast Linear Regression (Wind_Power + Photovoltaic)"

TRAIN_END = pd.Timestamp("2025-10-31 23:45:00")
VAL_END = pd.Timestamp("2025-11-30 23:45:00")

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## Load Forecast Dataset

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)

required_columns = ["time", *ALL_FEATURE_COLUMNS, TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

model_df = df[required_columns].copy()

print(f"Rows: {len(model_df):,}")
print(f"Time range: {model_df['time'].min()} to {model_df['time'].max()}")
print(f"Aggregate feature set: {AGGREGATE_FEATURE_COLUMNS}")
print(f"Component feature set: {COMPONENT_FEATURE_COLUMNS}")
display(model_df.head())
display(model_df[ALL_FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T)

## Chronological Split

- Train: January through October 2025
- Validation: November 2025
- Test: December 2025

In [ ]:
train_df = model_df[model_df["time"] <= TRAIN_END].copy()
val_df = model_df[(model_df["time"] > TRAIN_END) & (model_df["time"] <= VAL_END)].copy()
test_df = model_df[model_df["time"] > VAL_END].copy()

split_summary = pd.DataFrame(
    {
        "train": {"rows": len(train_df), "start": train_df["time"].min(), "end": train_df["time"].max()},
        "validation": {"rows": len(val_df), "start": val_df["time"].min(), "end": val_df["time"].max()},
        "test": {"rows": len(test_df), "start": test_df["time"].min(), "end": test_df["time"].max()},
    }
).T
display(split_summary)

y_train = train_df[TARGET_COLUMN]
y_val = val_df[TARGET_COLUMN]
y_test = test_df[TARGET_COLUMN]

X_train_aggregate = train_df[AGGREGATE_FEATURE_COLUMNS]
X_val_aggregate = val_df[AGGREGATE_FEATURE_COLUMNS]
X_test_aggregate = test_df[AGGREGATE_FEATURE_COLUMNS]

X_train_component = train_df[COMPONENT_FEATURE_COLUMNS]
X_val_component = val_df[COMPONENT_FEATURE_COLUMNS]
X_test_component = test_df[COMPONENT_FEATURE_COLUMNS]

## Helpers

In [ ]:
def make_linear_model():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def evaluate_model(model, model_name, datasets):
    rows = []
    for split_name, (X, y) in datasets.items():
        pred = model.predict(X)
        rows.append({"model": model_name, "split": split_name, **regression_metrics(y, pred)})
    return pd.DataFrame(rows)


def coefficient_table(model, model_name, feature_columns):
    estimator = model.named_steps["model"]
    return pd.DataFrame(
        {
            "model": model_name,
            "feature": feature_columns,
            "coefficient_per_1_std": estimator.coef_,
        }
    ).sort_values("coefficient_per_1_std", key=lambda values: values.abs(), ascending=False)

## Train Linear Regression Models

In [ ]:
aggregate_model = make_linear_model()
aggregate_model.fit(X_train_aggregate, y_train)

component_model = make_linear_model()
component_model.fit(X_train_component, y_train)

aggregate_datasets = {
    "train": (X_train_aggregate, y_train),
    "validation": (X_val_aggregate, y_val),
    "test": (X_test_aggregate, y_test),
}
component_datasets = {
    "train": (X_train_component, y_train),
    "validation": (X_val_component, y_val),
    "test": (X_test_component, y_test),
}

aggregate_metrics = evaluate_model(aggregate_model, AGGREGATE_MODEL_NAME, aggregate_datasets)
component_metrics = evaluate_model(component_model, COMPONENT_MODEL_NAME, component_datasets)
forecast_baseline_metrics = pd.concat([aggregate_metrics, component_metrics], ignore_index=True)

display(forecast_baseline_metrics.sort_values(["split", "RMSE"]))

forecast_coefficients = pd.concat(
    [
        coefficient_table(aggregate_model, AGGREGATE_MODEL_NAME, AGGREGATE_FEATURE_COLUMNS),
        coefficient_table(component_model, COMPONENT_MODEL_NAME, COMPONENT_FEATURE_COLUMNS),
    ],
    ignore_index=True,
)
display(forecast_coefficients)

## Test Prediction Plot

In [ ]:
test_predictions = test_df[["time", TARGET_COLUMN]].copy()
test_predictions["forecast_linear_wind_and_solar_pred"] = aggregate_model.predict(X_test_aggregate)
test_predictions["forecast_linear_wind_power_photovoltaic_pred"] = component_model.predict(X_test_component)
test_predictions["forecast_linear_wind_and_solar_residual"] = (
    test_predictions[TARGET_COLUMN] - test_predictions["forecast_linear_wind_and_solar_pred"]
)
test_predictions["forecast_linear_wind_power_photovoltaic_residual"] = (
    test_predictions[TARGET_COLUMN] - test_predictions["forecast_linear_wind_power_photovoltaic_pred"]
)

plot_window = test_predictions.head(7 * 24 * 4)

fig_pred, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(plot_window["time"], plot_window[TARGET_COLUMN], label="Actual", color="#111111", linewidth=1.8)
ax.plot(
    plot_window["time"],
    plot_window["forecast_linear_wind_and_solar_pred"],
    label="Forecast Linear Wind_And_Solar",
    color="#2166ac",
    alpha=0.85,
)
ax.plot(
    plot_window["time"],
    plot_window["forecast_linear_wind_power_photovoltaic_pred"],
    label="Forecast Linear Wind + PV",
    color="#1b9e77",
    alpha=0.85,
)
ax.set_title("Forecast Linear Regression Test Predictions: First 7 Days Of December")
ax.set_xlabel("Time")
ax.set_ylabel("Standardized price")
ax.legend()
fig_pred.autofmt_xdate()
fig_pred.tight_layout()
display(fig_pred)

## RMSE And R2 Comparison Charts

In [ ]:
split_order = ["train", "validation", "test"]
split_labels = ["Train", "Validation", "Test"]
feature_sets = ["Wind_And_Solar", "Wind_Power + Photovoltaic"]
colors = ["#2166ac", "#1b9e77"]

comparison_metrics = pd.concat(
    [
        aggregate_metrics.assign(feature_set="Wind_And_Solar"),
        component_metrics.assign(feature_set="Wind_Power + Photovoltaic"),
    ],
    ignore_index=True,
)


def plot_metric_comparison(metric, ylabel, title):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    x = np.arange(len(split_order))
    width = 0.36

    for offset_idx, (feature_set, color) in enumerate(zip(feature_sets, colors)):
        values = (
            comparison_metrics[comparison_metrics["feature_set"] == feature_set]
            .set_index("split")
            .loc[split_order, metric]
        )
        positions = x + (offset_idx - 0.5) * width
        bars = ax.bar(positions, values, width=width, label=feature_set, color=color)
        ax.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(split_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    return fig, ax


fig_rmse, ax = plot_metric_comparison(
    metric="RMSE",
    ylabel="RMSE",
    title="Forecast Linear Regression RMSE Comparison",
)
display(fig_rmse)

fig_r2, ax = plot_metric_comparison(
    metric="R2",
    ylabel="R2",
    title="Forecast Linear Regression R2 Comparison",
)
display(fig_r2)

## Save Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

aggregate_metrics.to_csv(OUTPUT_DIR / "linear_regression_forecast_wind_and_solar_metrics.csv", index=False)
component_metrics.to_csv(OUTPUT_DIR / "linear_regression_forecast_wind_power_photovoltaic_metrics.csv", index=False)
forecast_baseline_metrics.to_csv(OUTPUT_DIR / "forecast_linear_regression_metrics.csv", index=False)
forecast_coefficients.to_csv(OUTPUT_DIR / "forecast_linear_regression_coefficients.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "forecast_baseline_test_predictions.csv", index=False)

fig_pred.savefig(OUTPUT_DIR / "forecast_baseline_test_predictions_first_week.png", bbox_inches="tight")
fig_rmse.savefig(OUTPUT_DIR / "forecast_linear_regression_rmse_comparison.png", bbox_inches="tight")
fig_r2.savefig(OUTPUT_DIR / "forecast_linear_regression_r2_comparison.png", bbox_inches="tight")

print(f"Saved metrics and figures to: {OUTPUT_DIR}")